In [2]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pickle

from utils.dataset import CaptionDataset, caption_collate_fn
from models.model import CaptionGenerator


[nltk_data] Downloading package punkt to /Users/camelot/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
# Load vocab
with open('../data/vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

# Create Dataset and DataLoader
dataset = CaptionDataset(
    captions_file='../data/cleaned_captions.json',
    features_file='../data/image_features.json',
    vocab_file='../data/vocab.pkl'
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=caption_collate_fn)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
embed_size = 256
hidden_size = 512
num_layers = 1
vocab_size = len(vocab)
learning_rate = 3e-4
num_epochs = 5

# Model
model = CaptionGenerator(embed_size, hidden_size, vocab_size, num_layers).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [6]:
model.train()

for epoch in range(num_epochs):
    total_loss = 0

    for batch_idx, (features, captions, lengths) in enumerate(dataloader):
        features, captions = features.to(device), captions.to(device)

        # Inputs are all except last word, targets are all except first
        inputs = captions[:, :-1]
        targets = captions[:, 1:]

        outputs = model(features, inputs)

        loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"✅ Epoch [{epoch+1}/{num_epochs}] finished. Average Loss: {avg_loss:.4f}")


Epoch [1/5], Step [100/1265], Loss: 4.5330
Epoch [1/5], Step [200/1265], Loss: 4.1388
Epoch [1/5], Step [300/1265], Loss: 4.2444
Epoch [1/5], Step [400/1265], Loss: 4.0336
Epoch [1/5], Step [500/1265], Loss: 3.7143
Epoch [1/5], Step [600/1265], Loss: 3.5681
Epoch [1/5], Step [700/1265], Loss: 3.6285
Epoch [1/5], Step [800/1265], Loss: 3.5943
Epoch [1/5], Step [900/1265], Loss: 3.1974
Epoch [1/5], Step [1000/1265], Loss: 3.5916
Epoch [1/5], Step [1100/1265], Loss: 3.3629
Epoch [1/5], Step [1200/1265], Loss: 3.4636
✅ Epoch [1/5] finished. Average Loss: 3.8845
Epoch [2/5], Step [100/1265], Loss: 3.5618
Epoch [2/5], Step [200/1265], Loss: 3.2366
Epoch [2/5], Step [300/1265], Loss: 3.0808
Epoch [2/5], Step [400/1265], Loss: 3.1251
Epoch [2/5], Step [500/1265], Loss: 3.2979
Epoch [2/5], Step [600/1265], Loss: 3.0064
Epoch [2/5], Step [700/1265], Loss: 3.0313
Epoch [2/5], Step [800/1265], Loss: 3.1945
Epoch [2/5], Step [900/1265], Loss: 3.0371
Epoch [2/5], Step [1000/1265], Loss: 3.0977
Epoch

In [7]:
torch.save(model.state_dict(), '../outputs/caption_model.pth')
print("📦 Model saved to outputs/caption_model.pth")

📦 Model saved to outputs/caption_model.pth


In [8]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_bleu(reference, predicted):
    # reference: list of tokenized captions (ground truths)
    # predicted: list of tokens
    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference], predicted, smoothing_function=smoothie)


In [9]:
ref = "a dog is running through the field".split()
pred = "dog running through field".split()

bleu_score = evaluate_bleu(ref, pred)
print(f"BLEU: {bleu_score:.4f}")


BLEU: 0.0945
